> 在线讲义：[第 29 章 · router z-loss 与 Mixtral：MoE 训练怎么稳住，一个开源 MoE 长什么样](https://wikieden.github.io/beyond-transformer/lessons/29-router-zloss-mixtral.html) · 本书《超越 Transformer》

# 第 29 章 · router z-loss 与 Mixtral：MoE 训练怎么稳住，一个开源 MoE 长什么样

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/29-router-zloss-mixtral.html) · 仓库里 `beyond-transformer/notes/lessons/29-router-zloss-mixtral.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 bf16 误差 | 舍入 + exp | `to_bf16`、`bf16_error` | 73.7 → 73.5，18.13% |
| 第二步 router z-loss | 式 (5) | `router_z_loss(logits)` | 2.54 vs 955.0 |
| 第三步 参数分布 | 按部分数 | `mixtral_breakdown()` | 专家 96.6% |

系数 c_z = 0.001，把 router logit 压小；Mixtral 8×7B 的参数都花在哪。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `to_bf16`**：把 float32 的低 16 位抹掉（按最近舍入），模拟 bfloat16：只剩 8 位有效数字（7 位尾数 + 隐含的 1）。

In [ ]:
def to_bf16(x):
    """把 float32 的低 16 位抹掉（按最近舍入），模拟 bfloat16：只剩 8 位有效数字（7 位尾数 + 隐含的 1）。"""
    a = np.asarray(x, np.float32).view(np.uint32)
    a = (a + 0x7FFF + ((a >> 16) & 1)) & 0xFFFF0000
    return a.view(np.float32).astype(float)

**函数 `bf16_error`**：router logit 存成 bf16 后的绝对误差，以及 exp 之后的相对误差。

In [ ]:
def bf16_error(logits):
    """router logit 存成 bf16 后的绝对误差，以及 exp 之后的相对误差。"""
    l = np.asarray(logits, float)
    b = to_bf16(l)
    return np.abs(b - l), np.abs(np.exp(b - l) - 1)

**函数 `router_z_loss`**：ST-MoE 式 (5)：L_z = (1/B) Σ_i (log Σ_j e^{x_ij})²。

In [ ]:
def router_z_loss(logits):
    """ST-MoE 式 (5)：L_z = (1/B) Σ_i (log Σ_j e^{x_ij})²。"""
    x = np.atleast_2d(logits)
    m = x.max(1, keepdims=True)
    lse = (m + np.log(np.exp(x - m).sum(1, keepdims=True))).ravel()
    return float((lse**2).mean())

**函数 `mixtral_breakdown`**：小工具。

In [ ]:
def mixtral_breakdown(d=4096, d_ff=14336, n_exp=8, k=2, layers=32, vocab=32000, n_kv=8, d_head=128):
    attn = layers * (2 * d * d + 2 * d * n_kv * d_head)
    experts = layers * n_exp * 3 * d * d_ff
    router = layers * d * n_exp
    emb = 2 * vocab * d
    active = attn + router + emb + layers * k * 3 * d * d_ff
    return {"注意力": attn, "专家": experts, "router": router, "嵌入 + 输出": emb}, active

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 bf16 舍入误差随 logit 变大")
for lg in (0.37, 3.7, 37.3, 73.7):
    ab, rel = bf16_error([lg])
    print(f"  logit {lg}：bf16 存成 {to_bf16([lg])[0]}，误差 {ab[0]:.4f}，exp 之后相对误差 {rel[0]:.2%}")
print("\n第二步 router z-loss（c_z = 0.001）")
for name, lg in (("小 logit", [0.5, -0.2, 0.1, 0.3]), ("大 logit", [30.5, 29.8, 10.0, 5.0])):
    z = router_z_loss(lg)
    print(f"  {name} {lg}：L_z = {z:.4f}，乘 c_z 后 {0.001 * z:.4f}")

In [ ]:
print("\n第三步 Mixtral 8×7B 的参数都花在哪")
parts, active = mixtral_breakdown()
total = sum(parts.values())
for k, v in parts.items():
    print(f"  {k}：{v / 1e9:.2f}B（{v / total:.1%}）")
print(f"  总 {total / 1e9:.1f}B，每个 token 激活 {active / 1e9:.1f}B")

## 3. 代码里才看得到的

- **bf16 就是 float32 的高 16 位**：所以转换只要把低 16 位按“就近、平局取偶”舍入掉；这也是 bf16 和 float32 指数范围一样大的原因。
- **z-loss 的 log-sum-exp 也要稳定地算**：先减最大值再 exp（第 1 章），否则大 logit 时自己就先溢出了。

## 4. 自己改一改

- 把 `bf16_error` 的 logit 换成 300、1000，相对误差变成多少？
- 给 `mixtral_breakdown` 换成 16 个专家选 2，专家的占比和激活参数各是多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 八格尺子量大数：bf16 数越大刻度越粗，exp 一放大，概率差两成。
> router z-loss 是 log-sum-exp 的平方，系数千分之一：logit 压回零附近，概率不变、误差变小。
> Mixtral 九成七在专家，激活两个；专家不按主题分，按句法分，邻居常同路。